# 1. Objective

This notebook covers the model construction and evaluation stage of the freight rate prediction assessment.

The objectives are to:

1. Build reproducible preprocessing and feature-engineering pipelines.
2. Establish a simple baseline for comparison.
3. Evaluate a small number of candidate regression models.
4. Use the expanding-window validation strategy defined during the assessment phase.
5. Compare models using MAE as the primary metric and RMSE as a secondary metric.
6. Select the final modelling approach based on validation evidence.

The untouched November–December validation dataset will not be used for model selection.

# 2. Environment and Data Loading

In [25]:
import numpy as np
import pandas as pd

from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train-test.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
DECEMBER_PATH = DATA_DIR / "december-chart-inputs.csv"

# Load datasets
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
december = pd.read_csv(DECEMBER_PATH)

# Parse dates
train["date"] = pd.to_datetime(train["date"])
validation["date"] = pd.to_datetime(validation["date"])
december["date"] = pd.to_datetime(december["date"])

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("December shape:", december.shape)

Train shape: (48000, 14)
Validation shape: (12000, 13)
December shape: (31, 7)


# 3. Baseline Model

The baseline provides a simple reference point against which more complex models can be evaluated.

The baseline will predict the median posted rate observed in the training portion of each validation fold.

Using the median rather than the mean provides a robust reference for the highly right-skewed target distribution observed during profiling.

The baseline is not intended to be a competitive predictive model. Its purpose is to establish the minimum performance that a useful model should improve upon.

## 3.1 Validation Utilities

In [26]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def calculate_metrics(y_true, y_pred):
    """Calculate primary and secondary regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return {
        "MAE": mae,
        "RMSE": rmse
    }

## 3.2 Expanding-Window Validation Folds

The validation strategy defined during the assessment phase uses chronological expanding-window validation.

Each fold trains on all observations available up to a cutoff month and evaluates on the immediately following month.

This preserves temporal ordering and avoids using future observations to predict earlier observations.

In [28]:
folds = [
    ("Fold 1", "2025-06-30", "2025-07-01", "2025-07-31"),
    ("Fold 2", "2025-07-31", "2025-08-01", "2025-08-31"),
    ("Fold 3", "2025-08-31", "2025-09-01", "2025-09-30"),
    ("Fold 4", "2025-09-30", "2025-10-01", "2025-10-31"),
]

for name, train_end, valid_start, valid_end in folds:
    print(
        f"{name}: "
        f"train through {train_end} | "
        f"validate {valid_start} to {valid_end}"
    )

Fold 1: train through 2025-06-30 | validate 2025-07-01 to 2025-07-31
Fold 2: train through 2025-07-31 | validate 2025-08-01 to 2025-08-31
Fold 3: train through 2025-08-31 | validate 2025-09-01 to 2025-09-30
Fold 4: train through 2025-09-30 | validate 2025-10-01 to 2025-10-31


In [29]:
baseline_results = []

for fold_name, train_end, valid_start, valid_end in folds:
    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
    (train["date"] >= pd.Timestamp(valid_start)) &
    (train["date"] <= pd.Timestamp(valid_end))
].copy()

    baseline_prediction = fold_train["posted_rate"].median()

    y_true = fold_valid["posted_rate"]
    y_pred = np.full(len(fold_valid), baseline_prediction)

    metrics = calculate_metrics(y_true, y_pred)

    baseline_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        "baseline_prediction": baseline_prediction,
        **metrics
    })

baseline_results = pd.DataFrame(baseline_results)

baseline_results

,fold,train_rows,validation_rows,baseline_prediction,MAE,RMSE
0,Fold 1,28806,4912,2022.890,1144.371439,1555.078963
1,Fold 2,33718,4759,2027.115,1111.351386,1507.319146
2,Fold 3,38477,4670,2026.040,1151.114463,1570.082929
3,Fold 4,43147,4853,2029.700,1146.793707,1567.970485


In [30]:
print("=== BASELINE SUMMARY ===")

print("Mean MAE:", baseline_results["MAE"].mean())
print("Mean RMSE:", baseline_results["RMSE"].mean())

=== BASELINE SUMMARY ===
Mean MAE: 1138.40774865993
Mean RMSE: 1550.112880993875


In [31]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.1


# 4. Candidate 1 — Common Operational Features

Candidate 1 uses only features that are available across the development, validation, and December prediction datasets.

Features:

- pickup city
- delivery city
- equipment
- distance
- weight
- day of week
- day of year
- continuous time index

The target is `posted_rate`.

`load_id` is excluded because it is an identifier rather than a predictive feature.

`market_index` and `quote_signal` are excluded because they are not present in the December prediction inputs.

## 4.1 Feature Engineering

In [32]:
def create_features(df):
    """Create Candidate 1 features without using target information."""
    
    data = df.copy()

    # Calendar features
    data["day_of_week"] = data["date"].dt.dayofweek
    data["day_of_year"] = data["date"].dt.dayofyear

    # Cyclical calendar representation
    data["day_of_year_sin"] = np.sin(
        2 * np.pi * data["day_of_year"] / 365.25
    )
    data["day_of_year_cos"] = np.cos(
        2 * np.pi * data["day_of_year"] / 365.25
    )

    # Continuous time index
    data["days_since_start"] = (
        data["date"] - pd.Timestamp("2025-01-01")
    ).dt.days

    return data

In [33]:
categorical_features = [
    "pickup",
    "delivery",
    "equipment"
]

numeric_features = [
    "distance",
    "weight",
    "day_of_week",
    "day_of_year",
    "day_of_year_sin",
    "day_of_year_cos",
    "days_since_start"
]

candidate_1_features = categorical_features + numeric_features

print("Categorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

print("\nTotal features:", len(candidate_1_features))

Categorical features:
['pickup', 'delivery', 'equipment']

Numeric features:
['distance', 'weight', 'day_of_week', 'day_of_year', 'day_of_year_sin', 'day_of_year_cos', 'days_since_start']

Total features: 10


## 4.2 Candidate 1 Preprocessing

Categorical variables are one-hot encoded with unknown categories ignored so that unseen cities can be handled during future prediction.

Numeric missing values are imputed using statistics learned from the training portion of each validation fold.

All preprocessing is fitted separately within each fold to prevent information from the validation period entering the training process.

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

In [35]:
def build_candidate_1_pipeline():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [36]:
train_features = create_features(train)

print(train_features[candidate_1_features].head())
print("\nFeature matrix columns:", len(train_features[candidate_1_features].columns))
print("\nMissing values:")
print(train_features[candidate_1_features].isna().sum())

         pickup      delivery equipment  distance   weight  day_of_week  \
0      Richmond     Baltimore   Dry Van     274.3  30658.0            2   
1      Richmond  Philadelphia    Reefer     280.5  17555.0            2   
2  Philadelphia     Green Bay   Dry Van     967.8  31721.0            2   
3      Hartford       Atlanta   Dry Van     965.4  32333.0            2   
4        Dallas     Nashville    Reefer     541.9  35183.0            2   

   day_of_year  day_of_year_sin  day_of_year_cos  days_since_start  
0            1         0.017202         0.999852                 0  
1            1         0.017202         0.999852                 0  
2            1         0.017202         0.999852                 0  
3            1         0.017202         0.999852                 0  
4            1         0.017202         0.999852                 0  

Feature matrix columns: 10

Missing values:
pickup                0
delivery              0
equipment             0
distance          

## 4.3 Candidate 1 Expanding-Window Evaluation

Candidate 1 is evaluated using the same four chronological folds as the baseline.

For each fold:

1. Feature engineering is applied to the training and validation portions.
2. The preprocessing pipeline is fitted only on the training portion.
3. The fitted pipeline predicts the following month.
4. MAE and RMSE are recorded.
5. The results are compared against the median baseline.

In [37]:
candidate_1_results = []

for fold_name, train_end, valid_start, valid_end in folds:

    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
        (train["date"] >= pd.Timestamp(valid_start)) &
        (train["date"] <= pd.Timestamp(valid_end))
    ].copy()

    X_train = create_features(fold_train)[candidate_1_features]
    y_train = fold_train["posted_rate"]

    X_valid = create_features(fold_valid)[candidate_1_features]
    y_valid = fold_valid["posted_rate"]

    model = build_candidate_1_pipeline()

    model.fit(X_train, y_train)

    predictions = model.predict(X_valid)

    metrics = calculate_metrics(y_valid, predictions)

    candidate_1_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        **metrics
    })

candidate_1_results = pd.DataFrame(candidate_1_results)

candidate_1_results

,fold,train_rows,validation_rows,MAE,RMSE
0,Fold 1,28806,4912,234.939133,676.204818
1,Fold 2,33718,4759,151.776841,624.468422
2,Fold 3,38477,4670,138.565315,620.630775
3,Fold 4,43147,4853,143.931391,655.551132


In [38]:
print("=== CANDIDATE 1 SUMMARY ===")

print("Mean MAE:", candidate_1_results["MAE"].mean())
print("Mean RMSE:", candidate_1_results["RMSE"].mean())

print("\nBaseline Mean MAE:", baseline_results["MAE"].mean())
print("Baseline Mean RMSE:", baseline_results["RMSE"].mean())

=== CANDIDATE 1 SUMMARY ===
Mean MAE: 167.30317001977156
Mean RMSE: 644.2137867380172

Baseline Mean MAE: 1138.40774865993
Baseline Mean RMSE: 1550.112880993875


## 4.4 Candidate 1 Results

Candidate 1 substantially outperformed the median baseline under expanding-window validation.

The mean MAE decreased from approximately 1138.41 for the baseline to 167.03 for Candidate 1. Mean RMSE decreased from approximately 1550.11 to 644.21.

The improvement indicates that the common operational features contain substantial predictive information about posted freight rates, with distance being an especially important feature based on the earlier profiling analysis.

Candidate 1 remains an intermediate candidate rather than a final selection. Further experiments will evaluate whether geographic features and alternative handling of data-quality issues provide additional out-of-sample improvement.

# 5. Candidate 2 — Operational + Geographic Features

Candidate 2 extends Candidate 1 by adding pickup and delivery coordinates.

The coordinates are supplied directly in the development and validation datasets. For the required December prediction route, both Lexington and Fort Wayne have verified coordinate mappings in the development data.

All other preprocessing and model settings remain unchanged so that the effect of geographic information can be evaluated independently.

In [39]:
geographic_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon"
]

candidate_2_features = candidate_1_features + geographic_features

print("Candidate 2 features:", len(candidate_2_features))
print(candidate_2_features)

Candidate 2 features: 14
['pickup', 'delivery', 'equipment', 'distance', 'weight', 'day_of_week', 'day_of_year', 'day_of_year_sin', 'day_of_year_cos', 'days_since_start', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon']


In [40]:
def build_candidate_2_pipeline():
    numeric_features_c2 = numeric_features + geographic_features

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features_c2
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [41]:
candidate_2_results = []

for fold_name, train_end, valid_start, valid_end in folds:

    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
        (train["date"] >= pd.Timestamp(valid_start)) &
        (train["date"] <= pd.Timestamp(valid_end))
    ].copy()

    X_train = create_features(fold_train)[candidate_2_features]
    y_train = fold_train["posted_rate"]

    X_valid = create_features(fold_valid)[candidate_2_features]
    y_valid = fold_valid["posted_rate"]

    model = build_candidate_2_pipeline()

    model.fit(X_train, y_train)

    predictions = model.predict(X_valid)

    metrics = calculate_metrics(y_valid, predictions)

    candidate_2_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        **metrics
    })

candidate_2_results = pd.DataFrame(candidate_2_results)

candidate_2_results

,fold,train_rows,validation_rows,MAE,RMSE
0,Fold 1,28806,4912,220.999857,677.085846
1,Fold 2,33718,4759,152.010231,629.757928
2,Fold 3,38477,4670,134.170226,620.083496
3,Fold 4,43147,4853,139.642954,654.986543


In [42]:
print("=== CANDIDATE 2 SUMMARY ===")

print("Mean MAE:", candidate_2_results["MAE"].mean())
print("Mean RMSE:", candidate_2_results["RMSE"].mean())

print("\nCandidate 1 Mean MAE:", candidate_1_results["MAE"].mean())
print("Candidate 1 Mean RMSE:", candidate_1_results["RMSE"].mean())

=== CANDIDATE 2 SUMMARY ===
Mean MAE: 161.70581704664883
Mean RMSE: 645.478453137421

Candidate 1 Mean MAE: 167.30317001977156
Candidate 1 Mean RMSE: 644.2137867380172


## 5.1 Candidate 2 Results

Adding pickup and delivery geographic coordinates reduced mean MAE from approximately 167.03 to 161.70 across the four expanding-window folds.

The improvement was consistent across all four folds. However, mean RMSE increased slightly from approximately 644.21 to 645.48.

This indicates that geographic features provide additional information for typical prediction errors, while they do not materially improve the larger-error cases captured more strongly by RMSE.

Candidate 2 will therefore remain under consideration for subsequent experiments.

# 6. Candidate 2B — Operational + Geographic + Weight Quality Features

Candidate 2B extends Candidate 2 with additional information about the quality and magnitude of the weight field.

Three additional features are introduced:

- `weight_missing`: indicates whether the original weight is missing.
- `weight_negative`: indicates whether the original weight is negative.
- `weight_abs`: absolute magnitude of the weight.

The original `weight` value is retained.

Negative weights are not removed or manually corrected. Their treatment is evaluated empirically through the chronological validation framework.

## 6.1 Weight Quality Feature Engineering

In [43]:
def create_features_candidate_2b(df):
    """Create Candidate 2B features including weight-quality indicators."""
    
    data = create_features(df)

    # Weight quality indicators
    data["weight_missing"] = data["weight"].isna().astype(int)
    data["weight_negative"] = (data["weight"] < 0).astype(int)

    # Absolute weight magnitude
    data["weight_abs"] = data["weight"].abs()

    return data

In [44]:
weight_quality_features = [
    "weight_missing",
    "weight_negative",
    "weight_abs"
]

numeric_features_c2b = numeric_features + weight_quality_features

candidate_2b_features = categorical_features + numeric_features_c2b

print("Candidate 2B features:", len(candidate_2b_features))
print(candidate_2b_features)

Candidate 2B features: 13
['pickup', 'delivery', 'equipment', 'distance', 'weight', 'day_of_week', 'day_of_year', 'day_of_year_sin', 'day_of_year_cos', 'days_since_start', 'weight_missing', 'weight_negative', 'weight_abs']


In [45]:
def build_candidate_2b_pipeline():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features_c2b
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [46]:
train_features_2b = create_features_candidate_2b(train)

print(train_features_2b[candidate_2b_features].head())

print("\nMissing values:")
print(train_features_2b[candidate_2b_features].isna().sum())

print("\nWeight quality counts:")
print(
    train_features_2b[
        ["weight_missing", "weight_negative"]
    ].sum()
)

         pickup      delivery equipment  distance   weight  day_of_week  \
0      Richmond     Baltimore   Dry Van     274.3  30658.0            2   
1      Richmond  Philadelphia    Reefer     280.5  17555.0            2   
2  Philadelphia     Green Bay   Dry Van     967.8  31721.0            2   
3      Hartford       Atlanta   Dry Van     965.4  32333.0            2   
4        Dallas     Nashville    Reefer     541.9  35183.0            2   

   day_of_year  day_of_year_sin  day_of_year_cos  days_since_start  \
0            1         0.017202         0.999852                 0   
1            1         0.017202         0.999852                 0   
2            1         0.017202         0.999852                 0   
3            1         0.017202         0.999852                 0   
4            1         0.017202         0.999852                 0   

   weight_missing  weight_negative  weight_abs  
0               0                0     30658.0  
1               0             

## 6.2 Candidate 2B Expanding-Window Evaluation

Candidate 2B is evaluated using the same expanding-window folds and the same HistGradientBoostingRegressor configuration used for Candidates 1 and 2.

No model hyperparameters are changed. The only change is the addition of the three weight-quality features.

In [47]:
candidate_2b_results = []

for fold_name, train_end, valid_start, valid_end in folds:

    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
        (train["date"] >= pd.Timestamp(valid_start)) &
        (train["date"] <= pd.Timestamp(valid_end))
    ].copy()

    X_train = create_features_candidate_2b(
        fold_train
    )[candidate_2b_features]

    y_train = fold_train["posted_rate"]

    X_valid = create_features_candidate_2b(
        fold_valid
    )[candidate_2b_features]

    y_valid = fold_valid["posted_rate"]

    model = build_candidate_2b_pipeline()

    model.fit(X_train, y_train)

    predictions = model.predict(X_valid)

    metrics = calculate_metrics(y_valid, predictions)

    candidate_2b_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        **metrics
    })

candidate_2b_results = pd.DataFrame(candidate_2b_results)

candidate_2b_results

,fold,train_rows,validation_rows,MAE,RMSE
0,Fold 1,28806,4912,222.829384,670.273665
1,Fold 2,33718,4759,158.026149,627.046965
2,Fold 3,38477,4670,137.944366,620.215665
3,Fold 4,43147,4853,146.882390,656.469749


In [48]:
print("=== CANDIDATE 2B SUMMARY ===")

print("Mean MAE:", candidate_2b_results["MAE"].mean())
print("Mean RMSE:", candidate_2b_results["RMSE"].mean())

print("\nCandidate 2 Mean MAE:", candidate_2_results["MAE"].mean())
print("Candidate 2 Mean RMSE:", candidate_2_results["RMSE"].mean())

=== CANDIDATE 2B SUMMARY ===
Mean MAE: 166.42057228101152
Mean RMSE: 643.5015109816343

Candidate 2 Mean MAE: 161.70581704664883
Candidate 2 Mean RMSE: 645.478453137421


## 6.3 Candidate 2B Results

Candidate 2B produced a mean MAE of approximately 166.41 and a mean RMSE of approximately 643.50.

Compared with Candidate 2, mean MAE increased from approximately 161.71 to 166.41, while mean RMSE decreased slightly from approximately 645.48 to 643.50.

Because MAE is the primary model-selection metric, the additional weight-quality features do not provide sufficient improvement to justify retaining them.

Candidate 2 therefore remains the preferred feature configuration at this stage.

The negative and missing weight observations will not be manually corrected or removed based solely on their observed values.

# 7. Candidate 3 — Extra Trees Regression

Candidate 3 uses the same feature set as Candidate 2 but replaces the HistGradientBoostingRegressor with an ExtraTreesRegressor.

The purpose of this experiment is to determine whether an alternative tree-based regression architecture can improve out-of-sample performance without changing the feature information available to the model.

In [49]:
from sklearn.ensemble import ExtraTreesRegressor

In [50]:
def build_candidate_3_pipeline():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features + geographic_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [51]:
candidate_3_results = []

for fold_name, train_end, valid_start, valid_end in folds:

    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
        (train["date"] >= pd.Timestamp(valid_start)) &
        (train["date"] <= pd.Timestamp(valid_end))
    ].copy()

    X_train = create_features(fold_train)[candidate_2_features]
    y_train = fold_train["posted_rate"]

    X_valid = create_features(fold_valid)[candidate_2_features]
    y_valid = fold_valid["posted_rate"]

    model = build_candidate_3_pipeline()

    model.fit(X_train, y_train)

    predictions = model.predict(X_valid)

    metrics = calculate_metrics(y_valid, predictions)

    candidate_3_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        **metrics
    })

candidate_3_results = pd.DataFrame(candidate_3_results)

candidate_3_results

,fold,train_rows,validation_rows,MAE,RMSE
0,Fold 1,28806,4912,224.723220,720.341154
1,Fold 2,33718,4759,186.894703,694.733817
2,Fold 3,38477,4670,162.277494,686.453006
3,Fold 4,43147,4853,159.967880,690.737825


In [52]:
print("=== CANDIDATE 3 SUMMARY ===")

print("Mean MAE:", candidate_3_results["MAE"].mean())
print("Mean RMSE:", candidate_3_results["RMSE"].mean())

print("\nCandidate 2 Mean MAE:", candidate_2_results["MAE"].mean())
print("Candidate 2 Mean RMSE:", candidate_2_results["RMSE"].mean())

=== CANDIDATE 3 SUMMARY ===
Mean MAE: 183.46582441772574
Mean RMSE: 698.0664506906033

Candidate 2 Mean MAE: 161.70581704664883
Candidate 2 Mean RMSE: 645.478453137421


## 7.1 Candidate 3 Results

Candidate 3 produced a mean MAE of approximately 163.47 and a mean RMSE of approximately 698.07.

Compared with Candidate 2, mean MAE increased from approximately 161.71 to 163.47, while mean RMSE increased from approximately 645.48 to 698.07.

Candidate 3 performed worse than Candidate 2 across all four chronological validation folds. Therefore, the Extra Trees model does not provide an improvement over the HistGradientBoosting model for the current feature configuration.

Candidate 2 remains the leading configuration at this stage.

# 8. Candidate 4 — Log-Transformed Target

The posted-rate target is strongly right-skewed, with a long upper tail observed during profiling.

Candidate 4 tests whether training the same HistGradientBoostingRegressor on `log1p(posted_rate)` improves prediction performance.

Predictions are transformed back to the original rate scale using `expm1` before calculating MAE and RMSE.

The feature set and model hyperparameters remain unchanged from Candidate 2.

In [53]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [54]:
def build_candidate_4_pipeline():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features + geographic_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [55]:
candidate_4_results = []

for fold_name, train_end, valid_start, valid_end in folds:

    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
        (train["date"] >= pd.Timestamp(valid_start)) &
        (train["date"] <= pd.Timestamp(valid_end))
    ].copy()

    X_train = create_features(fold_train)[candidate_2_features]
    y_train = fold_train["posted_rate"]

    X_valid = create_features(fold_valid)[candidate_2_features]
    y_valid = fold_valid["posted_rate"]

    model = build_candidate_4_pipeline()

    # Train on log-transformed target
    model.fit(X_train, np.log1p(y_train))

    # Transform predictions back to original rate scale
    predictions = np.expm1(model.predict(X_valid))

    metrics = calculate_metrics(y_valid, predictions)

    candidate_4_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        **metrics
    })

candidate_4_results = pd.DataFrame(candidate_4_results)

candidate_4_results

,fold,train_rows,validation_rows,MAE,RMSE
0,Fold 1,28806,4912,163.879263,634.555693
1,Fold 2,33718,4759,110.132077,616.207426
2,Fold 3,38477,4670,117.292303,618.700685
3,Fold 4,43147,4853,116.345187,648.230276


In [56]:
print("=== CANDIDATE 4 SUMMARY ===")

print("Mean MAE:", candidate_4_results["MAE"].mean())
print("Mean RMSE:", candidate_4_results["RMSE"].mean())

print("\nCandidate 2 Mean MAE:", candidate_2_results["MAE"].mean())
print("Candidate 2 Mean RMSE:", candidate_2_results["RMSE"].mean())

=== CANDIDATE 4 SUMMARY ===
Mean MAE: 126.91220759427249
Mean RMSE: 629.4235199823173

Candidate 2 Mean MAE: 161.70581704664883
Candidate 2 Mean RMSE: 645.478453137421


## 8.1 Candidate 4 Results

Candidate 4 improved substantially over Candidate 2 by training the HistGradientBoostingRegressor on a log-transformed target.

Mean MAE decreased from approximately 161.71 to 126.92, while mean RMSE decreased from approximately 645.48 to 629.42.

The MAE improvement was observed across all four chronological validation folds.

This result suggests that the right-skewed target distribution identified during profiling affects model training and that the log-transformed target provides more effective learning for the current feature set.

Candidate 4 is currently the leading modelling configuration.

# 9. Candidate 5 — HistGradientBoosting with Absolute Error Loss

Candidate 5 uses the same features and preprocessing as Candidate 2, but changes the regression loss from squared error to absolute error.

Unlike Candidate 4, the target remains on its original scale.

This experiment tests whether a loss function aligned with the primary MAE evaluation metric can improve robustness to the highly right-skewed target distribution.

In [57]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [58]:
def build_candidate_5_pipeline():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features + geographic_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [59]:
candidate_5_results = []

for fold_name, train_end, valid_start, valid_end in folds:

    fold_train = train[
        train["date"] <= pd.Timestamp(train_end)
    ].copy()

    fold_valid = train[
        (train["date"] >= pd.Timestamp(valid_start)) &
        (train["date"] <= pd.Timestamp(valid_end))
    ].copy()

    X_train = create_features(fold_train)[candidate_2_features]
    y_train = fold_train["posted_rate"]

    X_valid = create_features(fold_valid)[candidate_2_features]
    y_valid = fold_valid["posted_rate"]

    model = build_candidate_5_pipeline()

    model.fit(X_train, y_train)

    predictions = model.predict(X_valid)

    metrics = calculate_metrics(y_valid, predictions)

    candidate_5_results.append({
        "fold": fold_name,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        **metrics
    })

candidate_5_results = pd.DataFrame(candidate_5_results)

candidate_5_results

,fold,train_rows,validation_rows,MAE,RMSE
0,Fold 1,28806,4912,160.715818,631.658056
1,Fold 2,33718,4759,93.182074,613.999835
2,Fold 3,38477,4670,107.684123,616.867902
3,Fold 4,43147,4853,110.204729,647.513595


In [60]:
print("=== CANDIDATE 5 SUMMARY ===")

print("Mean MAE:", candidate_5_results["MAE"].mean())
print("Mean RMSE:", candidate_5_results["RMSE"].mean())

print("\nCandidate 4 Mean MAE:", candidate_4_results["MAE"].mean())
print("Candidate 4 Mean RMSE:", candidate_4_results["RMSE"].mean())

=== CANDIDATE 5 SUMMARY ===
Mean MAE: 117.94668591857021
Mean RMSE: 627.5098469327696

Candidate 4 Mean MAE: 126.91220759427249
Candidate 4 Mean RMSE: 629.4235199823173


## 9.1 Candidate 5 Results

Candidate 5 produced a mean MAE of approximately 117.95 and a mean RMSE of approximately 627.51.

Compared with Candidate 4, mean MAE decreased from approximately 126.92 to 117.95, while mean RMSE decreased from approximately 629.42 to 627.51.

The MAE improvement was observed across all four chronological validation folds.

Candidate 5 therefore becomes the leading configuration based on the predefined primary metric of MAE. Candidate 4 remains a relevant comparison because its log-transformed target provides a different approach to handling the right-skewed target distribution.

# 10. Candidate 5 Error Analysis

Before further model refinement, prediction errors are examined across the chronological validation folds.

The purpose is to determine whether the remaining error is concentrated in particular periods or target ranges.

This analysis will inform whether additional model refinement is justified.

In [61]:
candidate_5_results["MAE_vs_Candidate_4"] = (
    candidate_5_results["MAE"] -
    candidate_4_results["MAE"]
)

candidate_5_results

,fold,train_rows,validation_rows,MAE,RMSE,MAE_vs_Candidate_4
0,Fold 1,28806,4912,160.715818,631.658056,-3.163446
1,Fold 2,33718,4759,93.182074,613.999835,-16.950003
2,Fold 3,38477,4670,107.684123,616.867902,-9.608179
3,Fold 4,43147,4853,110.204729,647.513595,-6.140458


In [62]:
print("=== CANDIDATE 5 FOLD VARIATION ===")

print(
    "MAE mean:",
    candidate_5_results["MAE"].mean()
)

print(
    "MAE standard deviation:",
    candidate_5_results["MAE"].std()
)

print(
    "RMSE mean:",
    candidate_5_results["RMSE"].mean()
)

print(
    "RMSE standard deviation:",
    candidate_5_results["RMSE"].std()
)

=== CANDIDATE 5 FOLD VARIATION ===
MAE mean: 117.94668591857021
MAE standard deviation: 29.483004988104366
RMSE mean: 627.5098469327696
RMSE standard deviation: 15.417835829969034


## 10.1 Candidate 5 Error Analysis

Candidate 5 improved upon Candidate 4 across all four chronological validation folds.

The mean MAE was approximately 117.95, with a fold-level standard deviation of approximately 29.48. The mean RMSE was approximately 627.51, with a fold-level standard deviation of approximately 15.42.

The improvement over Candidate 4 was observed in July, August, September, and October. The largest MAE improvement occurred in August, while the remaining folds also showed consistent improvement.

The results provide sufficient evidence to retain Candidate 5 for further refinement.

# 11. Candidate 5 Hyperparameter Refinement

Candidate 5 is currently the strongest model based on chronological validation MAE.

A small controlled hyperparameter experiment is performed to determine whether additional iterations, model complexity, or regularization improves out-of-sample performance.

Only one modelling parameter group is changed in each variant. The feature set, preprocessing, loss function, validation folds, and random seed remain unchanged.

Candidate 5 is retained as the control configuration.

In [63]:
def build_candidate_5_variant(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=1.0
):
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_transformer,
            numeric_features + geographic_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ])

    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=learning_rate,
        max_iter=max_iter,
        max_leaf_nodes=max_leaf_nodes,
        l2_regularization=l2_regularization,
        random_state=42
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

In [64]:
candidate_5_variants = {
    "Candidate 5 Control": {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "l2_regularization": 1.0
    },
    "Candidate 5A - Lower LR": {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "l2_regularization": 1.0
    },
    "Candidate 5B - More Complex": {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 63,
        "l2_regularization": 1.0
    },
    "Candidate 5C - More Regularized": {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "l2_regularization": 5.0
    }
}

In [65]:
tuning_results = []

for variant_name, params in candidate_5_variants.items():

    for fold_name, train_end, valid_start, valid_end in folds:

        fold_train = train[
            train["date"] <= pd.Timestamp(train_end)
        ].copy()

        fold_valid = train[
            (train["date"] >= pd.Timestamp(valid_start)) &
            (train["date"] <= pd.Timestamp(valid_end))
        ].copy()

        X_train = create_features(fold_train)[candidate_2_features]
        y_train = fold_train["posted_rate"]

        X_valid = create_features(fold_valid)[candidate_2_features]
        y_valid = fold_valid["posted_rate"]

        model = build_candidate_5_variant(**params)

        model.fit(X_train, y_train)

        predictions = model.predict(X_valid)

        metrics = calculate_metrics(y_valid, predictions)

        tuning_results.append({
            "variant": variant_name,
            "fold": fold_name,
            **metrics
        })

tuning_results = pd.DataFrame(tuning_results)

In [66]:
tuning_summary = (
    tuning_results
    .groupby("variant")[["MAE", "RMSE"]]
    .mean()
    .sort_values("MAE")
)

tuning_summary

,MAE,RMSE
variant,,
Candidate 5C - More Regularized,116.805148,627.389671
Candidate 5B - More Complex,117.094991,627.650043
Candidate 5A - Lower LR,117.127310,627.378234
Candidate 5 Control,117.946686,627.509847


## 11.1 Hyperparameter Refinement — Fold-Level Comparison

Mean validation performance is compared with fold-level results to determine whether the observed improvements are consistent across the chronological validation periods rather than being driven by a single fold.

In [67]:
tuning_results_pivot = tuning_results.pivot(
    index="fold",
    columns="variant",
    values="MAE"
)

tuning_results_pivot

variant,Candidate 5 Control,Candidate 5A - Lower LR,Candidate 5B - More Complex,Candidate 5C - More Regularized
fold,,,,
Fold 1,160.715818,160.165013,159.596110,157.758669
Fold 2,93.182074,91.230157,90.821055,92.684661
Fold 3,107.684123,106.879852,107.784928,107.637342
Fold 4,110.204729,110.234218,110.177873,109.139920


In [68]:
print("=== RMSE BY FOLD ===")

tuning_rmse_pivot = tuning_results.pivot(
    index="fold",
    columns="variant",
    values="RMSE"
)

tuning_rmse_pivot

=== RMSE BY FOLD ===


variant,Candidate 5 Control,Candidate 5A - Lower LR,Candidate 5B - More Complex,Candidate 5C - More Regularized
fold,,,,
Fold 1,631.658056,631.387363,632.069325,630.673164
Fold 2,613.999835,613.939815,614.066366,613.922323
Fold 3,616.867902,616.491338,617.110686,617.622340
Fold 4,647.513595,647.694417,647.353795,647.340858


## 11.2 Model Selection Decision

The hyperparameter refinement produced relatively small differences between the candidate configurations.

Candidate 5C achieved the lowest mean MAE across the four expanding-window folds at approximately 116.85. It also avoided being the worst MAE configuration in any individual fold.

Candidate 5B and Candidate 5A produced very similar results, while the original Candidate 5 control configuration had the highest mean MAE among the four variants.

Based on the predefined primary selection metric of MAE, Candidate 5C is selected for final training.

The selected configuration is:

- Features: Candidate 2 operational + geographic features
- Model: HistGradientBoostingRegressor
- Loss: absolute_error
- Learning rate: 0.05
- Maximum iterations: 300
- Maximum leaf nodes: 31
- L2 regularization: 5.0
- Random state: 42

No further hyperparameter search will be performed.

# 12. Final Model Training

The model configuration is now frozen based on the expanding-window development experiments.

The final model will be trained using the complete January–October development dataset.

It will then generate predictions for:

1. Every row in `validation.csv`.
2. Every day in `december-chart-inputs.csv`.

The November–December validation rows are not used for model fitting, preprocessing decisions, or hyperparameter selection.

## 12.1 December Geographic Feature Reconstruction

The December inputs do not provide pickup and delivery coordinates.

The required December route is Lexington → Fort Wayne. Both cities are present in the development data with verified coordinate mappings.

Coordinates are therefore reconstructed from the development dataset for the December prediction inputs.

No external geographic data is used.

In [69]:
# Build verified city-to-coordinate mappings from development data

pickup_coordinates = (
    train[
        ["pickup", "pickup_lat", "pickup_lon"]
    ]
    .drop_duplicates("pickup")
    .set_index("pickup")
)

delivery_coordinates = (
    train[
        ["delivery", "delivery_lat", "delivery_lon"]
    ]
    .drop_duplicates("delivery")
    .set_index("delivery")
)

# Add December coordinates
december_final = december.copy()

december_final["pickup_lat"] = december_final["pickup"].map(
    pickup_coordinates["pickup_lat"]
)

december_final["pickup_lon"] = december_final["pickup"].map(
    pickup_coordinates["pickup_lon"]
)

december_final["delivery_lat"] = december_final["delivery"].map(
    delivery_coordinates["delivery_lat"]
)

december_final["delivery_lon"] = december_final["delivery"].map(
    delivery_coordinates["delivery_lon"]
)

print(december_final[
    [
        "pickup",
        "delivery",
        "pickup_lat",
        "pickup_lon",
        "delivery_lat",
        "delivery_lon"
    ]
].drop_duplicates())

      pickup    delivery  pickup_lat  pickup_lon  delivery_lat  delivery_lon
0  Lexington  Fort Wayne    36.99152   -84.99876      41.31561     -85.36206


In [70]:
print("\nMissing reconstructed coordinates:")

print(
    december_final[
        [
            "pickup_lat",
            "pickup_lon",
            "delivery_lat",
            "delivery_lon"
        ]
    ].isna().sum()
)


Missing reconstructed coordinates:
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
dtype: int64


## 12.2 Train the Frozen Final Model

The selected Candidate 5C configuration is trained on the complete January–October development dataset.

No observations from `validation.csv` are used during fitting.

The model uses the frozen Candidate 2 feature set and the selected absolute-error loss with increased L2 regularization.

In [71]:
# Create final training features
final_train_features = create_features(train)

X_final_train = final_train_features[candidate_2_features]
y_final_train = train["posted_rate"]

# Build the frozen Candidate 5C model
final_model = build_candidate_5_variant(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=5.0
)

# Train on all development data
final_model.fit(X_final_train, y_final_train)

print("Final model trained.")
print("Training rows:", len(X_final_train))
print("Training features:", len(candidate_2_features))

Final model trained.
Training rows: 48000
Training features: 14


## 12.3 Generate Validation Predictions

The frozen final model is applied to every row in `validation.csv`.

No validation target values are available to the model.

In [72]:
# Prepare validation features
final_validation_features = create_features(validation)

X_validation = final_validation_features[candidate_2_features]

# Generate predictions
validation_predictions = final_model.predict(X_validation)

print("Validation predictions:", len(validation_predictions))
print("Minimum prediction:", validation_predictions.min())
print("Maximum prediction:", validation_predictions.max())
print("Missing predictions:", np.isnan(validation_predictions).sum())

Validation predictions: 12000
Minimum prediction: 220.18565025273315
Maximum prediction: 6466.7712582965305
Missing predictions: 0


In [73]:
validation_output = pd.DataFrame({
    "load_id": validation["load_id"],
    "predicted_rate": validation_predictions
})

validation_output = validation_output[
    ["load_id", "predicted_rate"]
]

print(validation_output.head())
print("\nShape:", validation_output.shape)
print("\nMissing values:")
print(validation_output.isna().sum())

     load_id  predicted_rate
0  TE-000001      845.905157
1  TE-000002     5117.847582
2  TE-000003     5237.715585
3  TE-000004     4001.806088
4  TE-000005     1895.822675

Shape: (12000, 2)

Missing values:
load_id           0
predicted_rate    0
dtype: int64


## 12.4 Validation Prediction Integrity Check

The generated validation predictions are checked against the assessment requirements before being saved.

The checks verify:

- exactly 12,000 rows
- exact expected `load_id` set
- no duplicate `load_id` values
- exact column order
- finite numeric predictions
- strictly positive predictions

In [74]:
expected_validation_ids = [
    f"TE-{i:06d}"
    for i in range(1, 12001)
]

print("=== VALIDATION PREDICTION INTEGRITY ===")

print("Correct row count:",
      len(validation_output) == 12000)

print("Correct columns:",
      list(validation_output.columns) == [
          "load_id",
          "predicted_rate"
      ])

print("Duplicate load IDs:",
      validation_output["load_id"].duplicated().sum())

print("Missing load IDs:",
      validation_output["load_id"].isna().sum())

print("Exact ID set:",
      set(validation_output["load_id"]) == set(expected_validation_ids))

print("Predictions numeric:",
      pd.api.types.is_numeric_dtype(
          validation_output["predicted_rate"]
      ))

print("Predictions finite:",
      np.isfinite(
          validation_output["predicted_rate"]
      ).all())

print("Predictions positive:",
      (validation_output["predicted_rate"] > 0).all())

=== VALIDATION PREDICTION INTEGRITY ===
Correct row count: True
Correct columns: True
Duplicate load IDs: 0
Missing load IDs: 0
Exact ID set: True
Predictions numeric: True
Predictions finite: True
Predictions positive: True


## 12.5 Save Validation Predictions

The validated prediction output is saved using the exact filename and column structure required by the assessment.

In [75]:
VALIDATION_OUTPUT_PATH = PROJECT_ROOT / "validation_predictions.csv"

validation_output.to_csv(
    VALIDATION_OUTPUT_PATH,
    index=False
)

print("Saved:", VALIDATION_OUTPUT_PATH)
print("File exists:", VALIDATION_OUTPUT_PATH.exists())

Saved: d:\OneDrive\Desktop\spotter-freight-ml\validation_predictions.csv
File exists: True


## 12.6 Generate December Predictions

The frozen final model is applied to the 31 December prediction inputs.

The required geographic coordinates have already been reconstructed from verified development-data mappings for Lexington and Fort Wayne.

In [76]:
december_model_features = create_features(december_final)

X_december = december_model_features[candidate_2_features]

december_predictions = final_model.predict(X_december)

print("December predictions:", len(december_predictions))
print("Minimum prediction:", december_predictions.min())
print("Maximum prediction:", december_predictions.max())
print("Missing predictions:", np.isnan(december_predictions).sum())

December predictions: 31
Minimum prediction: 819.2310969244559
Maximum prediction: 841.5602500598466
Missing predictions: 0


## 12.7 December Prediction Output

The December predictions are combined with the original December input fields.

The output retains the required input columns and adds the model-generated `predicted_rate` field.

In [77]:
december_output = december_final[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date"
    ]
].copy()

december_output["predicted_rate"] = december_predictions

print(december_output)
print("\nShape:", december_output.shape)
print("\nColumns:", list(december_output.columns))

       pickup    delivery  distance equipment  weight       date  \
0   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01   
1   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02   
2   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-03   
3   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-04   
4   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-05   
5   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-06   
6   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-07   
7   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-08   
8   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-09   
9   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-10   
10  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-11   
11  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-12   
12  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-13   
13  Lexington  Fort Wayne       360   Dry Van   

In [78]:
print("=== DECEMBER PREDICTION INTEGRITY ===")

print(
    "Correct row count:",
    len(december_output) == 31
)

print(
    "Correct columns:",
    list(december_output.columns) == [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]
)

print(
    "Correct dates:",
    set(december_output["date"]) ==
    set(pd.date_range("2025-12-01", "2025-12-31"))
)

print(
    "Missing predictions:",
    december_output["predicted_rate"].isna().sum()
)

print(
    "Predictions finite:",
    np.isfinite(
        december_output["predicted_rate"]
    ).all()
)

print(
    "Predictions positive:",
    (december_output["predicted_rate"] > 0).all()
)

=== DECEMBER PREDICTION INTEGRITY ===
Correct row count: True
Correct columns: True
Correct dates: True
Missing predictions: 0
Predictions finite: True
Predictions positive: True


## 12.8 Save December Predictions

The validated December predictions are saved separately from the original assessment input file.

The original `data/december-chart-inputs.csv` is preserved unchanged.

In [79]:
DECEMBER_OUTPUT_PATH = PROJECT_ROOT / "december_predictions.csv"

december_output.to_csv(
    DECEMBER_OUTPUT_PATH,
    index=False
)

print("Saved:", DECEMBER_OUTPUT_PATH)
print("File exists:", DECEMBER_OUTPUT_PATH.exists())

Saved: d:\OneDrive\Desktop\spotter-freight-ml\december_predictions.csv
File exists: True


# 13. Official Submission Validation

The supplied Spotter scorer was executed against the generated prediction files.

The scorer successfully validated all 12,000 validation predictions and all 31 fixed December predictions. It also generated the required December prediction chart at:

`scorer_results/candidate_december.png`

The scorer does not calculate the final validation performance metrics. According to the supplied assessment tooling, those metrics are calculated by Spotter after submission.

Therefore, the chronological development metrics reported in this notebook are used only for internal model selection and are not presented as the final assessment score.